In [108]:
print("naveen")

naveen


In [3]:
%pwd

'd:\\medical_chatbot'

In [2]:
import os
os.chdir("../")

In [10]:
from langchain.document_loaders import PyPDFLoader,DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


d:\medical_chatbot\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
def load_pdf_files(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    docs = loader.load()
    return docs

In [12]:
extracted_data= load_pdf_files("data")

In [13]:
len(extracted_data)

637

In [14]:
from typing import List
from langchain.schema import Document

# filter only source and page content from extracted_data

In [15]:
def filter_to_minimal_docs(docs:List[Document])->List[Document]:
    minimal_docs:List[Document]=[]
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source":src}
            )
        )
    return minimal_docs


In [16]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [17]:
len(minimal_docs)

637

# splitting into chunks

In [18]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )

    text_chunks = text_splitter.split_documents(minimal_docs)
    return text_chunks

In [19]:
text_chunks = text_split(minimal_docs)
print(f"total Number of Chunks: {len(text_chunks)}")

total Number of Chunks: 5859


# embedding model

In [20]:
from langchain.embeddings import HuggingFaceEmbeddings

In [21]:
def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name,
    )
    return embeddings

In [22]:
embeddings = download_embeddings()

C:\Users\NAVEEN\AppData\Local\Temp\ipykernel_58276\574443911.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [23]:
vector = embeddings.embed_query("hello")

In [24]:
len(vector)

384

In [25]:
from dotenv import load_dotenv
import os


load_dotenv()


True

In [26]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")


In [27]:
groq_api = os.getenv("GROQ_API_KEY")

In [30]:
from pinecone import Pinecone


In [31]:
pineconedb = Pinecone(api_key=PINECONE_API_KEY)

In [32]:
from pinecone import ServerlessSpec

index_name = "medical-chatbot"

In [33]:
if not pineconedb.has_index(index_name):
    pineconedb.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

In [34]:
index = pineconedb.Index(index_name)

In [35]:
from langchain_pinecone import PineconeVectorStore

In [36]:
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)

In [37]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [38]:
retrieved_docs = retriever.invoke("what is Acne?")

In [39]:
print(retrieved_docs[0].page_content)

GALE ENCYCLOPEDIA OF MEDICINE 226
Acne
GEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26


In [40]:
from  langchain_groq import ChatGroq

In [42]:
model = ChatGroq(model="openai/gpt-oss-120b",api_key=groq_api)

In [45]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
system_prompt = (
    "you are an Medical assistent for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. if you dont't know the answer, say that you "
    "don't Know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

In [47]:
prompt= ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),

    ]
)

In [48]:
question_answer_chain = create_stuff_documents_chain(model,prompt)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)

In [52]:
response= rag_chain.invoke({"input":"how pimple is  caused?"})


In [53]:
print(response['answer'])

A pimple forms when a hair follicle becomes damaged, allowing excess sebum, dead skin cells, and bacteria (Propionibacterium acnes) to leak into surrounding tissue. The excess sebum—often increased by higher androgen levels during puberty—mixes with sticky skin cells to create a clogged pore (comedone). When the clogged follicle is invaded by bacteria, inflammation occurs, producing the visible pimple.
